# Diffusion Planner from scratch

This notebook independently rebuilds a compact Diffusion Planner. It does not import the repository package: each tensor contract, diffusion operation, encoder, denoiser, loss, sampler, and guidance energy is defined below. The implementation follows Zheng et al. (ICLR 2025), while using synthetic structured inputs instead of nuPlan.

## 1. The planning task

The model jointly generates the ego vehicle trajectory and predicts nearby vehicles. A state is `(x, y, cos(yaw), sin(yaw))`. The first state is the observed current state; the remaining states are future predictions. Conditioning contains agent history, static objects, and vectorized lanes.

In [ ]:
import math
import torch
from torch import nn

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
history_steps, future_steps = 8, 12
neighbors, static_objects, lanes, lane_points = 4, 4, 8, 12
participants = 1 + neighbors
trajectory_dim = (future_steps + 1) * 4
device

## 2. Structured input contract

The official implementation consumes vectorized tensors rather than images. Agent history is `(B, neighbors, history, 11)`: four pose values, velocity, dimensions, and a three-way type encoding. Lane vectors have twelve features including geometry and traffic-light information. Zero rows are padding and become attention masks.

In [ ]:
batch = 2
inputs = {
    'ego_current_state': torch.randn(batch, 4, device=device),
    'neighbor_agents_past': torch.randn(batch, neighbors, history_steps, 11, device=device),
    'static_objects': torch.randn(batch, static_objects, 10, device=device),
    'lanes': torch.randn(batch, lanes, lane_points, 12, device=device),
}
target = torch.randn(batch, participants, future_steps + 1, 4, device=device)
{key: tuple(value.shape) for key, value in inputs.items()}, tuple(target.shape)

## 3. Forward diffusion

A VP schedule gradually changes a clean trajectory `x0` into Gaussian noise. The closed-form sample is `xt = alpha(t) * x0 + sigma(t) * noise`, so training can choose one random continuous timestep without simulating every previous step.

In [ ]:
class VPSchedule:
    def __init__(self, beta_min=0.1, beta_max=20.0):
        self.beta_min, self.beta_max = beta_min, beta_max

    def alpha(self, t):
        log_alpha = -0.25 * t.square() * (self.beta_max - self.beta_min) - 0.5 * self.beta_min * t
        return log_alpha.exp()

    def sigma(self, t):
        return (1 - self.alpha(t).square()).clamp_min(1e-6).sqrt()

    def add_noise(self, x0, t):
        noise = torch.randn_like(x0)
        return self.alpha(t).view(-1, 1, 1) * x0 + self.sigma(t).view(-1, 1, 1) * noise

schedule = VPSchedule()
flat_target = target.reshape(batch, participants, trajectory_dim)
t = torch.rand(batch, device=device).clamp(1e-4, 1 - 1e-4)
noisy_target = schedule.add_noise(flat_target, t)
tuple(noisy_target.shape), t

## 4. MLP-Mixer for sparse vectors

Agent histories and lane polylines are sparse vector sequences. The paper uses MLP-Mixer blocks: one MLP mixes positions along the vector, then another mixes feature channels. Mean pooling produces one dense token per agent or lane.

In [ ]:
class MixerBlock(nn.Module):
    def __init__(self, token_count, channels):
        super().__init__()
        self.norm_tokens = nn.LayerNorm(channels)
        self.tokens = nn.Sequential(nn.Linear(token_count, token_count), nn.GELU(), nn.Linear(token_count, token_count))
        self.norm_channels = nn.LayerNorm(channels)
        self.channels = nn.Sequential(nn.Linear(channels, 2 * channels), nn.GELU(), nn.Linear(2 * channels, channels))

    def forward(self, x):
        update = self.tokens(self.norm_tokens(x).transpose(1, 2)).transpose(1, 2)
        x = x + update
        return x + self.channels(self.norm_channels(x))

class VectorEncoder(nn.Module):
    def __init__(self, feature_count, vector_count, hidden=64):
        super().__init__()
        self.project = nn.Linear(feature_count, hidden)
        self.mix = MixerBlock(vector_count, hidden)
        self.out = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, 96))

    def forward(self, x):
        x = self.mix(self.project(x))
        return self.out(x.mean(dim=1))


## 5. Context encoder

Each history, static-object, and lane group becomes a token. A transformer encoder lets those tokens exchange information. Padding masks prevent empty vectors from contributing, while the resulting context will be used as keys and values by the trajectory decoder.

In [ ]:
class ContextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.agents = VectorEncoder(11, history_steps)
        self.lanes = VectorEncoder(12, lane_points)
        self.static = nn.Sequential(nn.Linear(10, 64), nn.GELU(), nn.Linear(64, 96))
        layer = nn.TransformerEncoderLayer(96, 4, batch_first=True, norm_first=True)
        self.fusion = nn.TransformerEncoder(layer, 2)

    def forward(self, values):
        agent_tokens = self.agents(values['neighbor_agents_past'])
        lane_tokens = self.lanes(values['lanes'])
        static_tokens = self.static(values['static_objects'])
        context = torch.cat([agent_tokens, static_tokens, lane_tokens], dim=1)
        mask = torch.cat([
            values['neighbor_agents_past'].abs().sum(dim=(-1, -2)) == 0,
            values['static_objects'].abs().sum(dim=-1) == 0,
            values['lanes'].abs().sum(dim=(-1, -2)) == 0,
        ], dim=1)
        return self.fusion(context, src_key_padding_mask=mask), mask

encoder = ContextEncoder().to(device)
context, context_mask = encoder(inputs)
tuple(context.shape), tuple(context_mask.shape)

## 6. Timestep and participant embeddings

The denoiser receives one token per participant. Sinusoidal timestep features describe the noise level, and a learned embedding distinguishes the ego token from neighbor tokens.

In [ ]:
def timestep_embedding(t, dimension):
    half = dimension // 2
    frequencies = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    values = t[:, None] * frequencies[None]
    embedding = torch.cat([values.cos(), values.sin()], dim=-1)
    return torch.cat([embedding, embedding[:, :1]], dim=-1)[:, :dimension]

class AdaDiTBlock(nn.Module):
    def __init__(self, hidden=96):
        super().__init__()
        self.norm = nn.LayerNorm(hidden, elementwise_affine=False)
        self.attn = nn.MultiheadAttention(hidden, 4, batch_first=True)
        self.cross = nn.MultiheadAttention(hidden, 4, batch_first=True)
        self.ffn = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, 4 * hidden), nn.GELU(), nn.Linear(4 * hidden, hidden))
        self.ada = nn.Linear(hidden, 6 * hidden)

    def forward(self, x, context, condition, mask):
        shift, scale, gate, _, _, _ = self.ada(condition).chunk(6, dim=-1)
        normalized = self.norm(x) * (1 + scale[:, None]) + shift[:, None]
        update = self.attn(normalized, normalized, normalized, key_padding_mask=mask)[0]
        x = x + gate[:, None] * update
        x = x + self.cross(x, context, context)[0]
        return x + self.ffn(x)


## 7. DiT trajectory denoiser

The noised flattened trajectory is projected to participant tokens. Each block performs participant self-attention and context cross-attention. The output is an estimate of the clean `x0`, matching the paper's training objective `||mu_theta(xt,t,C) - x0||^2`.

In [ ]:
class TrajectoryDiT(nn.Module):
    def __init__(self, hidden=96, depth=2):
        super().__init__()
        self.input = nn.Linear(trajectory_dim, hidden)
        self.participant = nn.Embedding(2, hidden)
        self.time = nn.Sequential(nn.Linear(hidden, 4 * hidden), nn.SiLU(), nn.Linear(4 * hidden, hidden))
        self.blocks = nn.ModuleList(AdaDiTBlock(hidden) for _ in range(depth))
        self.output = nn.Sequential(nn.LayerNorm(hidden), nn.Linear(hidden, trajectory_dim))

    def forward(self, noisy, t, context, neighbor_mask):
        b, p, _ = noisy.shape
        ids = torch.zeros(p, dtype=torch.long, device=noisy.device); ids[1:] = 1
        x = self.input(noisy) + self.participant(ids)[None]
        condition = self.time(timestep_embedding(t, x.shape[-1]))
        mask = torch.cat([torch.zeros(b, 1, dtype=torch.bool, device=noisy.device), neighbor_mask[:, :p-1]], dim=1)
        for block in self.blocks:
            x = block(x, context, condition, mask)
        return self.output(x)

denoiser = TrajectoryDiT().to(device)
prediction = denoiser(noisy_target, t, context, context_mask)
tuple(prediction.shape)

## 8. One diffusion training step

Sample a timestep, add noise, predict the clean trajectory, and minimize mean squared error. In a real experiment, targets are normalized ego-centric nuPlan states and batches come from preprocessed scenario files.

In [ ]:
optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(denoiser.parameters()), lr=1e-4)
def training_step(values, clean):
    context, mask = encoder(values)
    t = torch.rand(clean.shape[0], device=clean.device).clamp(1e-4, 1 - 1e-4)
    noisy = schedule.add_noise(clean.reshape(clean.shape[0], participants, trajectory_dim), t)
    estimate = denoiser(noisy, t, context, mask)
    return (estimate - clean.reshape(clean.shape[0], participants, trajectory_dim)).square().mean()

loss = training_step(inputs, target)
optimizer.zero_grad(); loss.backward(); optimizer.step()
float(loss)

## 9. Sampling and the current-state constraint

Start from noise and repeatedly move toward the denoiser's clean estimate. This educational sampler uses a simple deterministic interpolation instead of the official repository's DPM-Solver. The current ego and neighbor states are inserted after every update, so the generated future is anchored to the observed scene.

In [ ]:
@torch.no_grad()
def sample(values, steps=8):
    context, mask = encoder(values)
    state = torch.randn(values['ego_current_state'].shape[0], participants, trajectory_dim, device=device)
    current = torch.cat([values['ego_current_state'][:, None], values['neighbor_agents_past'][:, :, -1, :4]], dim=1)
    times = torch.linspace(1.0, 1e-3, steps + 1, device=device)
    for index in range(steps):
        t = times[index].expand(state.shape[0])
        clean = denoiser(state, t, context, mask)
        state = clean + (times[index + 1] / times[index]) * (state - clean)
        state = state.reshape(state.shape[0], participants, future_steps + 1, 4)
        state[:, :, 0] = current
        state = state.reshape(state.shape[0], participants, trajectory_dim)
    return state.reshape(state.shape[0], participants, future_steps + 1, 4)[:, :, 1:]

prediction = sample(inputs)
tuple(prediction.shape)

## 10. Training-free classifier guidance

The paper treats preferences as differentiable energies. Collision avoidance penalizes ego-neighbor overlap; comfort penalizes excessive acceleration. During sampling, compute `grad_x E(mu_theta(xt))` and subtract it from the denoiser direction. Combining energies changes behavior without retraining the diffusion model.

In [ ]:
def collision_energy(trajectory, radius=2.0):
    ego = trajectory[:, :1, :, :2]
    others = trajectory[:, 1:, :, :2]
    distance = (ego - others).square().sum(dim=-1).sqrt().clamp_min(1e-4)
    return (radius - distance).relu().square().mean(dim=(1, 2))

def comfort_energy(trajectory, limit=3.0):
    position = trajectory[:, 0, :, :2]
    acceleration = position[:, 2:] - 2 * position[:, 1:-1] + position[:, :-2]
    return (acceleration.square().sum(dim=-1) - limit**2).relu().mean(dim=-1)

def guided_prediction(state, t, context, mask, scale=0.1):
    state = state.detach().requires_grad_(True)
    clean = denoiser(state, t, context, mask)
    trajectory = clean.reshape(clean.shape[0], participants, future_steps + 1, 4)
    energy = (collision_energy(trajectory) + comfort_energy(trajectory)).sum()
    gradient = torch.autograd.grad(energy, state)[0]
    return clean - scale * gradient

context, mask = encoder(inputs)
state = torch.randn(batch, participants, trajectory_dim, device=device, requires_grad=True)
guided = guided_prediction(state, torch.full((batch,), 0.5, device=device), context, mask)
tuple(guided.shape)

## 11. What this study leaves out

The official repository adds nuPlan scenario preprocessing, state normalization statistics, richer map/agent feature construction, a route encoder, DPM-Solver sampling, and closed-loop runners. Those are engineering layers around the central ideas demonstrated here. Do not interpret synthetic loss or open-loop samples as closed-loop driving performance.